# Frontier League Multivariable Regression

This notebook estimates multivariable OLS models for Frontier League team performance using a clean econometrics workflow. Update the variable selector near the top, rerun from top to bottom, and the formulas, diagnostics, plots, and interpretations will all refresh automatically.


In [ ]:
from pathlib import Path
from io import StringIO
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.regressionplots import influence_plot
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13

SIGNIFICANCE_LEVEL = 0.05
generated_figures = {}


## 1. Imports, Data Load, and Variable Selection

Why this section exists:
- Econometric work is only as reliable as the data preparation and variable definitions behind it.
- By resolving user-friendly aliases like `OPS` and `WinPct` to the actual CSV columns, the notebook stays easy to edit without breaking the regression code.
- This notebook is focused on the main multivariable model only, so it does not include reverse regressions.


In [ ]:
DATA_PATH = Path("frontier_league_master_stats_2021_2025_FULL.csv")

DEPENDENT_VAR = "WinPct"

INDEPENDENT_VARS = [
    "OPS",
    "FIP",
]

REMOVE_OUTLIERS = False

EXPORT_RESULTS = False
EXPORT_DIR = Path("exports/frontier_multivariable_regression")

ALIAS_MAP = {
    "WinPct": "Pitching_W-L%",
    "OPS": "Batting_OPS",
    "OBP": "Batting_OBP",
    "SLG": "Batting_SLG",
    "BA": "Batting_BA",
    "HR": "Batting_HR",
    "RBI": "Batting_RBI",
    "Runs": "Batting_R",
    "Runs_Per_Game": "Batting_R/G",
    "ERA": "Pitching_ERA",
    "RA9": "Pitching_RA9",
    "WHIP": "Pitching_WHIP",
    "FIP": "FIP",
    "H9": "Pitching_H9",
    "HR9": "Pitching_HR9",
    "BB9": "Pitching_BB9",
    "SO9": "Pitching_SO9",
    "SO_W": "Pitching_SO/W",
}


def resolve_variable(var_name, columns, alias_map):
    """Resolve a user-friendly variable name to a real CSV column."""
    if var_name in columns:
        return var_name

    mapped_name = alias_map.get(var_name)
    if isinstance(mapped_name, str) and mapped_name in columns:
        return mapped_name

    lower_lookup = {column.lower(): column for column in columns}
    if var_name.lower() in lower_lookup:
        return lower_lookup[var_name.lower()]
    if isinstance(mapped_name, str) and mapped_name.lower() in lower_lookup:
        return lower_lookup[mapped_name.lower()]

    available_aliases = ", ".join(sorted(alias_map))
    raise KeyError(
        f"Could not resolve '{var_name}'. Use an exact CSV column name or one of these aliases: {available_aliases}"
    )


def user_label(selected_name, resolved_name):
    return selected_name if selected_name == resolved_name else f"{selected_name} -> {resolved_name}"


def formula_term_name(column_name):
    return f'Q("{column_name}")'


def render_info_text(frame):
    buffer = StringIO()
    frame.info(buf=buffer)
    return buffer.getvalue()


def load_and_clean_data(path):
    """Load the CSV, remove duplicates, and coerce non-Team columns to numeric where possible."""
    if not path.exists():
        raise FileNotFoundError(f"Could not find the data file: {path.resolve()}")

    frame = pd.read_csv(path).copy()
    original_rows = len(frame)
    frame = frame.drop_duplicates().copy()
    duplicates_removed = original_rows - len(frame)

    for column in frame.columns:
        if column == "Team":
            frame[column] = frame[column].astype("string")
        else:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    return frame, duplicates_removed


def validate_selection(frame, dependent_var, independent_vars, alias_map):
    if not independent_vars:
        raise ValueError("INDEPENDENT_VARS must contain at least one regressor.")

    resolved_dep = resolve_variable(dependent_var, frame.columns, alias_map)
    resolved_indep = [resolve_variable(var, frame.columns, alias_map) for var in independent_vars]

    if len(set(resolved_indep)) != len(resolved_indep):
        raise ValueError("The selected independent variables resolve to duplicate columns.")
    if resolved_dep in resolved_indep:
        raise ValueError("The dependent variable cannot also appear as an independent variable.")

    return resolved_dep, resolved_indep


def build_analysis_frame(frame, dependent_col, independent_cols, metadata_cols=None):
    metadata_cols = metadata_cols or ["Year", "Team"]
    available_metadata = [column for column in metadata_cols if column in frame.columns]
    keep_cols = list(dict.fromkeys(available_metadata + [dependent_col] + independent_cols))
    analysis = frame[keep_cols].copy()
    analysis = analysis.dropna(subset=[dependent_col] + independent_cols).reset_index(drop=True)
    return analysis


def build_formula(dependent_col, independent_cols):
    rhs = " + ".join([formula_term_name(column) for column in independent_cols])
    return f'{formula_term_name(dependent_col)} ~ {rhs}'


def fit_ols(formula, analysis_frame):
    return smf.ols(formula=formula, data=analysis_frame).fit()


def make_coef_table(result):
    param_names = list(result.model.exog_names)
    conf_int = np.asarray(result.conf_int())
    table = pd.DataFrame(
        {
            "coef": np.asarray(result.params),
            "std_err": np.asarray(result.bse),
            "t_stat": np.asarray(result.tvalues),
            "p_value": np.asarray(result.pvalues),
            "ci_lower": conf_int[:, 0],
            "ci_upper": conf_int[:, 1],
        },
        index=param_names,
    )
    return table.rename_axis("term").round(4)


def make_fit_summary(result):
    return pd.Series(
        {
            "R-squared": result.rsquared,
            "Adjusted R-squared": result.rsquared_adj,
            "F-statistic": np.nan if result.fvalue is None else float(result.fvalue),
            "F-statistic p-value": np.nan if result.f_pvalue is None else float(result.f_pvalue),
            "Observations": int(result.nobs),
        }
    )


def heterosk_tests(result):
    """Run Breusch-Pagan and White tests to check whether the error variance is constant."""
    residuals = result.resid
    exog = result.model.exog

    bp_lm, bp_lm_pvalue, bp_f, bp_f_pvalue = het_breuschpagan(residuals, exog)
    white_lm, white_lm_pvalue, white_f, white_f_pvalue = het_white(residuals, exog)

    test_table = pd.DataFrame(
        [
            {
                "Test": "Breusch-Pagan",
                "LM Statistic": bp_lm,
                "LM p-value": bp_lm_pvalue,
                "F Statistic": bp_f,
                "F p-value": bp_f_pvalue,
            },
            {
                "Test": "White",
                "LM Statistic": white_lm,
                "LM p-value": white_lm_pvalue,
                "F Statistic": white_f,
                "F p-value": white_f_pvalue,
            },
        ]
    )

    heteroskedasticity_detected = bool((test_table["LM p-value"] < SIGNIFICANCE_LEVEL).any())
    return test_table.round(4), heteroskedasticity_detected


def compute_vif_table(analysis_frame, independent_cols):
    """Variance Inflation Factors diagnose how much collinearity inflates variance for each regressor."""
    design_matrix = sm.add_constant(analysis_frame[independent_cols], has_constant="add")
    vif_rows = []

    for index, column in enumerate(design_matrix.columns):
        if column == "const":
            continue
        vif_value = variance_inflation_factor(design_matrix.values, index)
        if vif_value > 10:
            flag = "VIF > 10 (severe)"
        elif vif_value > 5:
            flag = "VIF > 5 (moderate)"
        else:
            flag = "Acceptable"
        vif_rows.append({"variable": column, "VIF": vif_value, "flag": flag})

    return pd.DataFrame(vif_rows).round(4)


def describe_r_squared(r_squared):
    if r_squared < 0.25:
        return "fairly weak"
    if r_squared < 0.50:
        return "modest"
    if r_squared < 0.75:
        return "fairly strong"
    return "very strong"


def interpret_term(selected_name, resolved_name, result, dependent_name, alpha=0.05):
    term_name = formula_term_name(resolved_name)
    coefficient = float(result.params[list(result.model.exog_names).index(term_name)])
    p_value = float(result.pvalues[list(result.model.exog_names).index(term_name)])

    if coefficient > 0:
        direction = f"Higher {selected_name} is associated with higher {dependent_name}"
    elif coefficient < 0:
        direction = f"Higher {selected_name} is associated with lower {dependent_name}"
    else:
        direction = f"{selected_name} does not change the fitted value of {dependent_name}"

    significance = "statistically significant" if p_value < alpha else "not statistically significant"
    magnitude = (
        f"A one-unit increase in {selected_name} changes {dependent_name} by about {coefficient:.4f}, "
        "holding the other selected variables constant."
    )
    return f"{direction}. The estimated coefficient is {coefficient:.4f} and the relationship is {significance} (p = {p_value:.4f}). {magnitude}"


In [ ]:
df, duplicates_removed = load_and_clean_data(DATA_PATH)
resolved_dep, resolved_indep = validate_selection(df, DEPENDENT_VAR, INDEPENDENT_VARS, ALIAS_MAP)
analysis_df = build_analysis_frame(df, resolved_dep, resolved_indep)

selected_model_columns = [resolved_dep] + resolved_indep
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

display(Markdown("### Resolved Variable Selection"))
print(f"Dependent variable: {user_label(DEPENDENT_VAR, resolved_dep)}")
print("Independent variables:")
for selected_name, resolved_name in zip(INDEPENDENT_VARS, resolved_indep):
    print(f" - {user_label(selected_name, resolved_name)}")

print(f"\nRows after duplicate removal: {len(df)}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Rows available for estimation after dropping missing model values: {len(analysis_df)}")

display(Markdown("### DataFrame Info"))
print(render_info_text(df))

display(Markdown("### Missing Value Summary"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

display(Markdown("### Numeric Columns"))
print(numeric_columns)

display(Markdown("### Preview of the Analysis Sample"))
display(analysis_df.head())


## 3. Descriptive Statistics

Why this section exists:
- Descriptive statistics show the scale and dispersion of each variable before formal modeling.
- Correlation and covariance matrices help identify broad relationships and possible multicollinearity.


In [ ]:
descriptive_stats = analysis_df[selected_model_columns].describe().T.round(4)
correlation_matrix = analysis_df[selected_model_columns].corr().round(4)
covariance_matrix = analysis_df[selected_model_columns].cov().round(4)

display(Markdown("### Summary Statistics"))
display(descriptive_stats)

display(Markdown("### Correlation Matrix"))
display(correlation_matrix)

display(Markdown("### Covariance Matrix"))
display(covariance_matrix)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
ax.set_title("Correlation Heatmap")
generated_figures["correlation_heatmap"] = fig
plt.show()

pair_grid = sns.pairplot(analysis_df[selected_model_columns], corner=True, diag_kind="hist")
pair_grid.fig.suptitle("Scatterplot Matrix", y=1.02)
generated_figures["scatterplot_matrix"] = pair_grid.fig
plt.show()


## 4. Regression, 5. Heteroskedasticity Tests, and 6. Multicollinearity Tests

Why these tests exist:
- OLS estimates the average relationship between the dependent variable and the selected regressors.
- Heteroskedasticity tests ask whether the error variance is constant. If it is not, the usual OLS standard errors can be misleading, so robust standard errors are often preferred.
- VIF and correlation review diagnose multicollinearity, which makes it harder to cleanly separate the effect of one regressor from another.

How to read the main regression table:
- `coef`: the estimated change in the dependent variable from a one-unit change in a regressor, holding the other selected variables constant.
- `std_err`: the standard error of the coefficient estimate. Smaller values mean the estimate is more precise; larger values mean more uncertainty.
- `t_stat`: the coefficient divided by its standard error. Larger absolute values usually provide stronger evidence against a zero effect.
- `p_value`: the probability of seeing a result this extreme if the true coefficient were actually zero. Small p-values, often below 0.05, are commonly treated as statistically significant evidence.
- `ci_lower` and `ci_upper`: the confidence interval bounds. If the interval does not include zero, that usually matches statistical significance at the corresponding level.
- `R-squared`: the share of variation in the dependent variable explained by the model.
- `Adjusted R-squared`: a version of R-squared that penalizes adding regressors that do not improve the model much.
- `F-statistic`: a test of whether the regressors are jointly useful in explaining the dependent variable.


In [ ]:
formula = build_formula(resolved_dep, resolved_indep)
model = fit_ols(formula, analysis_df)
term_label_map = {formula_term_name(resolved): selected for selected, resolved in zip(INDEPENDENT_VARS, resolved_indep)}

coef_table = make_coef_table(model).rename(index=term_label_map)
fit_summary = make_fit_summary(model).round(4)

display(Markdown(f"### Model Formula\n`{formula}`"))
display(Markdown("### OLS Coefficient Table"))
display(coef_table)

display(Markdown("### Overall Model Fit"))
display(fit_summary.to_frame("value"))
display(
    Markdown(
        """### How To Read These Results
- `coef`: estimated change in `WinPct` from a one-unit increase in a regressor, holding the other selected regressors constant.
- `std_err`: the uncertainty around each coefficient estimate. Smaller values usually mean more precision.
- `t_stat`: coefficient divided by its standard error. Larger absolute values usually indicate stronger evidence.
- `p_value`: if this is small, often below 0.05, the regressor is commonly treated as statistically significant.
- `ci_lower` and `ci_upper`: the confidence interval for the coefficient. If zero is outside the interval, that usually matches significance.
- `R-squared`: the share of variation in `WinPct` explained by the model.
- `Adjusted R-squared`: a version of `R-squared` that penalizes regressors that add little explanatory power.
- `F-statistic`: tests whether the selected regressors are jointly useful overall.
"""
    )
)

heterosk_table, heteroskedasticity_detected = heterosk_tests(model)
display(Markdown("### Heteroskedasticity Tests"))
display(heterosk_table)
print("Null hypothesis: the regression errors have constant variance (homoskedasticity).")
print("Alternative hypothesis: the regression errors have non-constant variance (heteroskedasticity).")

if heteroskedasticity_detected:
    robust_model = model.get_robustcov_results(cov_type="HC1")
    robust_coef_table = make_coef_table(robust_model).rename(index=term_label_map)
    print("At least one test rejects the null at the 5% level, so HC1 robust standard errors are reported for inference.")
    display(Markdown("### HC1 Robust Coefficient Table"))
    display(robust_coef_table)
else:
    robust_model = None
    robust_coef_table = None
    print("Neither test rejects the null at the 5% level, so the classical OLS standard errors are retained.")

vif_table = compute_vif_table(analysis_df, resolved_indep)
vif_table["variable"] = vif_table["variable"].replace({resolved: selected for selected, resolved in zip(INDEPENDENT_VARS, resolved_indep)})
display(Markdown("### Variance Inflation Factors (VIF)"))
display(vif_table)
display(
    Markdown(
        """### How To Read The VIF Table
- `VIF` measures how strongly each regressor overlaps with the other selected regressors.
- A VIF near 1 suggests very little multicollinearity.
- A VIF above 5 is often treated as a warning sign of moderate multicollinearity.
- A VIF above 10 is often treated as a sign of severe multicollinearity.
- High VIF values do not automatically bias OLS coefficients, but they can make coefficients unstable and harder to interpret.
"""
    )
)

print("Multicollinearity means the regressors move together so strongly that coefficient uncertainty increases.")
print("High VIF values do not automatically bias OLS coefficients, but they make p-values and coefficient interpretation less stable.")


## 7. Residual Diagnostics and 8. Outlier / Influence Tests

Why this section exists:
- Residual plots help evaluate whether the linear model assumptions look reasonable.
- Normality checks matter most for small-sample inference because severe non-normality can distort hypothesis tests.
- Influence diagnostics show whether a small number of team-seasons are disproportionately shaping the fitted regression line.


In [ ]:
reporting_model = robust_model if robust_model is not None else model

fitted_values = model.fittedvalues
residuals = model.resid
influence = model.get_influence()
studentized_residuals = influence.resid_studentized_internal
sqrt_abs_studentized = np.sqrt(np.abs(studentized_residuals))

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

sns.scatterplot(x=fitted_values, y=residuals, ax=axes[0, 0], s=90)
axes[0, 0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set_title("Residuals vs. Fitted")
axes[0, 0].set_xlabel("Fitted Values")
axes[0, 0].set_ylabel("Residuals")

sns.histplot(residuals, kde=True, ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("Histogram of Residuals")
axes[0, 1].set_xlabel("Residual")

sm.qqplot(residuals, line="45", fit=True, ax=axes[1, 0])
axes[1, 0].set_title("QQ Plot of Residuals")

sns.scatterplot(x=fitted_values, y=sqrt_abs_studentized, ax=axes[1, 1], s=90)
sns.regplot(x=fitted_values, y=sqrt_abs_studentized, scatter=False, lowess=True, ax=axes[1, 1], color="crimson")
axes[1, 1].set_title("Scale-Location Plot")
axes[1, 1].set_xlabel("Fitted Values")
axes[1, 1].set_ylabel("Sqrt(|Studentized Residual|)")

fig.tight_layout()
generated_figures["residual_diagnostics"] = fig
plt.show()

jb_stat, jb_pvalue, jb_skew, jb_kurtosis = jarque_bera(residuals)
print("Jarque-Bera normality test")
print("Null hypothesis: the residuals follow a normal distribution.")
print("Alternative hypothesis: the residuals are not normally distributed.")
print(f"JB statistic: {jb_stat:.4f}")
print(f"JB p-value: {jb_pvalue:.4f}")
print(f"Residual skewness: {jb_skew:.4f}")
print(f"Residual kurtosis: {jb_kurtosis:.4f}")
if jb_pvalue < SIGNIFICANCE_LEVEL:
    print("Interpretation: reject normality at the 5% level, so normal-theory inference should be treated cautiously.")
else:
    print("Interpretation: do not reject normality at the 5% level, so the residual distribution does not show a strong normality violation.")

cooks_distance = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag
n_obs = len(analysis_df)
n_params = len(model.params)
cook_threshold = 4 / n_obs
leverage_threshold = 2 * n_params / n_obs

fig, ax = plt.subplots(figsize=(13, 9))
influence_plot(model, ax=ax, criterion="cooks")
ax.set_title("Influence Plot")
generated_figures["influence_plot"] = fig
plt.show()

metadata_cols = [column for column in ["Year", "Team"] if column in analysis_df.columns]
influence_df = analysis_df[metadata_cols].copy()
influence_df["CooksDistance"] = cooks_distance
influence_df["Leverage"] = leverage
influence_df["StudentizedResidual"] = studentized_residuals
influence_df["HighCooks"] = influence_df["CooksDistance"] > cook_threshold
influence_df["HighLeverage"] = influence_df["Leverage"] > leverage_threshold

flagged_df = influence_df.loc[influence_df["HighCooks"] | influence_df["HighLeverage"]].copy()
flagged_df = flagged_df.sort_values(["HighCooks", "HighLeverage", "CooksDistance", "Leverage"], ascending=[False, False, False, False])

print(f"Cook's distance threshold: {cook_threshold:.4f}")
print(f"Leverage threshold: {leverage_threshold:.4f}")
print("High leverage observations have unusual predictor values and can pull the fitted line toward themselves.")
print("High Cook's distance observations are influential because removing them would noticeably change the fitted model.")

if flagged_df.empty:
    print("No observations exceed the default Cook's distance or leverage thresholds.")
else:
    display(Markdown("### Potentially Influential Team-Seasons"))
    display(flagged_df.round(4))


## 9. Optional Robustness Check, 10. Interpretation, and Export

Why this section exists:
- A robustness check asks whether the core findings survive after removing unusually influential observations.
- Automatic interpretation turns the regression output into readable text, but you should still combine it with baseball knowledge and broader context.


In [ ]:
comparison_table = None
interpretation_lines = []

if REMOVE_OUTLIERS:
    influential_index = influence_df.index[influence_df["HighCooks"]].tolist()
    if influential_index:
        trimmed_df = analysis_df.drop(index=influential_index).reset_index(drop=True)
        trimmed_model = fit_ols(formula, trimmed_df)
        comparison_table = pd.DataFrame(
            {
                "original_coef": model.params,
                "trimmed_coef": trimmed_model.params,
                "difference": trimmed_model.params - model.params,
            }
        ).round(4)
        comparison_table.index = [term_label_map.get(index, index) for index in comparison_table.index]

        display(Markdown("### Robustness Check After Removing High Cook's Distance Observations"))
        display(comparison_table)
        print(f"Original R-squared: {model.rsquared:.4f}")
        print(f"Trimmed R-squared: {trimmed_model.rsquared:.4f}")
    else:
        print("REMOVE_OUTLIERS = True, but no high Cook's distance observations were identified.")
else:
    print("Outlier-removal robustness check is skipped because REMOVE_OUTLIERS = False.")

for selected_name, resolved_name in zip(INDEPENDENT_VARS, resolved_indep):
    interpretation_lines.append(
        f"- {interpret_term(selected_name, resolved_name, reporting_model, DEPENDENT_VAR, SIGNIFICANCE_LEVEL)}"
    )

interpretation_lines.append(
    f"- The model R-squared is {model.rsquared:.3f}, so the selected regressors explain about {model.rsquared:.1%} of the variation in {DEPENDENT_VAR}. This is {describe_r_squared(model.rsquared)} explanatory power."
)

if robust_model is not None:
    interpretation_lines.append(
        "- HC1 robust standard errors were used for inference because the heteroskedasticity tests suggested non-constant error variance."
    )

display(Markdown("### Automatic Interpretation"))
display(Markdown("\n".join(interpretation_lines)))

if EXPORT_RESULTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    descriptive_stats.to_csv(EXPORT_DIR / "descriptive_stats.csv")
    correlation_matrix.to_csv(EXPORT_DIR / "correlation_matrix.csv")
    covariance_matrix.to_csv(EXPORT_DIR / "covariance_matrix.csv")
    coef_table.to_csv(EXPORT_DIR / "ols_coefficients.csv")
    fit_summary.to_frame("value").to_csv(EXPORT_DIR / "model_fit_summary.csv")
    heterosk_table.to_csv(EXPORT_DIR / "heteroskedasticity_tests.csv", index=False)
    vif_table.to_csv(EXPORT_DIR / "vif_table.csv", index=False)
    influence_df.to_csv(EXPORT_DIR / "influence_diagnostics.csv", index=False)
    flagged_df.to_csv(EXPORT_DIR / "flagged_influential_observations.csv", index=False)

    if robust_coef_table is not None:
        robust_coef_table.to_csv(EXPORT_DIR / "robust_coefficients_hc1.csv")
    if comparison_table is not None:
        comparison_table.to_csv(EXPORT_DIR / "outlier_robustness_comparison.csv")

    with open(EXPORT_DIR / "ols_summary.txt", "w", encoding="utf-8") as handle:
        handle.write(model.summary().as_text())

    if robust_model is not None:
        with open(EXPORT_DIR / "robust_summary_hc1.txt", "w", encoding="utf-8") as handle:
            handle.write(robust_model.summary().as_text())

    with open(EXPORT_DIR / "interpretation.md", "w", encoding="utf-8") as handle:
        handle.write("\n".join(interpretation_lines))

    for figure_name, figure in generated_figures.items():
        figure.savefig(EXPORT_DIR / f"{figure_name}.png", dpi=300, bbox_inches="tight")

    print(f"Exported tables, summaries, and plots to: {EXPORT_DIR.resolve()}")
else:
    print("EXPORT_RESULTS = False, so no files were written to disk.")
